# **DISTIL Usage Example: Twitter**

Here, we show how to use DISTIL to perform active learning on sentiment analysis tasks using a BERT model with the Twitter Eval sentiment dataset. This notebook can be easily executed on Google Colab.

## Installation and Imports

In [6]:

# Standard imports
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import string
import unicodedata
import math

# Imports for datasets
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import DataLoader, TensorDataset, Dataset, Subset, ConcatDataset
from collections import Counter
from tqdm.auto import tqdm

# Imports for active learning
sys.path.append('distil/')
from distil.active_learning_strategies import BADGE, EntropySampling, LeastConfidenceSampling, MarginSampling, RandomSampling   # All active learning strategies showcased in this example
from distil.utils.utils import LabeledToUnlabeledDataset                       # A utility wrapper class that removes labels from labeled PyTorch dataset objects

## Preparing AGN_eval

tweet_eval is a dataset from Hugging Face that is used for sentiment analysis/sentiment classification. The dataset consists of 59899 English tweets from Twitter users and is split into 3 splits for training (45615), validation (2000), and testing (12284). The labels are as follows: 0 = negative, 1 = neutral, 2 = positive. The dataset can be found here: https://huggingface.co/datasets/tweet_eval

Here, we preprocess each data instance (a tweet) by making all words lowercase, removing punctuation, converting to ASCII format, and removing stopwords. Stopwords are commonly used words in the English language that usually do not provide any useful information for sentiment classification.

In [7]:


dataset = load_dataset("fancyzhx/ag_news")

train_dataset = dataset['train']
test_dataset = dataset['test']

# Text preprocessing functions
def to_lower(ex):
    ex['text'] = ex['text'].lower()
    return ex

def remove_punc(ex):
    ex['text'] = ''.join(x for x in ex['text'] if x not in string.punctuation)
    return ex

def to_ascii(ex):
    ex['text'] = ''.join(x for x in unicodedata.normalize('NFD', ex['text']) if unicodedata.category(x) != 'Mn')
    return ex

stopwords = [ "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at", "be", "because", 
             "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down", "during",
             "each", "few", "for", "from", "further", "had", "has", "have", "having", "he", "he'd", "he'll", "he's", "her", "here", 
             "here's", "hers", "herself", "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into",
             "is", "it", "it's", "its", "itself", "let's", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or",
             "other", "ought", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "she'd", "she'll", "she's", "should", 
             "so", "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves", "then", "there", "there's",
             "these", "they", "they'd", "they'll", "they're", "they've", "this", "those", "through", "to", "too", "under", "until", "up",
             "very", "was", "we", "we'd", "we'll", "we're", "we've", "were", "what", "what's", "when", "when's", "where", "where's",
             "which", "while", "who", "who's", "whom", "why", "why's", "with", "would", "you", "you'd", "you'll", "you're", "you've",
             "your", "yours", "yourself", "yourselves" ]

def remove_stopwords(ex):
    ex['text'] = ' '.join(x for x in ex['text'].split() if x not in stopwords)
    return ex

def process_data(data):
    data1 = data.map(remove_punc)
    data2 = data1.map(to_ascii)
    data3 = data2.map(to_lower)
    data4 = data3.map(remove_stopwords)
    return data4

# Apply processing
training_data = process_data(train_dataset)
testing_data = process_data(test_dataset)


## Tokenization and Pre-Active Learning

Now that the text has been pre-processed, tokenization of each instance can be performed to produce usable input for BERT. Here, we tokenize the pre-processed tweets using the Huggingface transformers package. We leverage the AutoTokenizer module with the bert-base-uncased configuration to do so. We then make sure that the dataset objects return PyTorch tensors by setting the format as "torch." 

Next, to create an active learning instance, we separate separate the training dataset into an initial labeled subset and an initial unlabeled pool. We also define the number of active learning rounds to perform and the active learning budget for each round.

In [ ]:
from transformers import AutoTokenizer, get_scheduler, BertConfig, AdamW, BertModel

configuration = BertConfig()
setattr(configuration, 'l1', 512)
setattr(configuration, 'num_classes', 4)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_func(ex):
    return tokenizer(ex['text'], padding='max_length', truncation=True)

# Removes unnecessary columns and renames columns to match the model specification
train1 = training_data.map(tokenize_func, batched=True)
test1 = testing_data.map(tokenize_func, batched=True)
train2 = train1.remove_columns("text")
test2 = test1.remove_columns("text")
training_dataset = train2.rename_column("label", "labels")
testing_dataset = test2.rename_column("label", "labels")

training_dataset.set_format("torch")
testing_dataset.set_format("torch")

# Define the number of active learning rounds to conduct, the budget, and the number of classes in AGN dataset
nclasses = 4
n_rounds = 6
budget = 300

train_size_pc     = 200
test_size_pc      = 1000
unlabeled_size_pc = 5000

# Ensure reproducibility
np.random.seed(42)

# Generate random index lists for subset creation
full_train_subset = []
full_unlabeled_subset = []
full_test_subset = []

for class_num in range(nclasses):
    train_idx_pool = []
    for train_idx in range(len(training_dataset)):
        if training_dataset[train_idx]["labels"].item() == class_num:
            train_idx_pool.append(train_idx)
    test_idx_pool = []
    for test_idx in range(len(testing_dataset)):
        if testing_dataset[test_idx]["labels"].item() == class_num:
            test_idx_pool.append(test_idx)

    # Get a random subset from each pool
    print(f"Class {class_num} - Training pool size: {len(train_idx_pool)}, Testing pool size: {len(test_idx_pool)}")
    train_idx_subset = np.random.choice(train_idx_pool, size=train_size_pc, replace = False).tolist()
    remaining_idx = list(set(train_idx_pool) - set(train_idx_subset))
    unlabeled_idx_subset = np.random.choice(remaining_idx, size=unlabeled_size_pc, replace = False).tolist()
    test_idx_subset = np.random.choice(test_idx_pool, size=test_size_pc, replace = False).tolist()
    full_train_subset.extend(train_idx_subset)
    full_unlabeled_subset.extend(unlabeled_idx_subset)
    full_test_subset.extend(test_idx_subset)

# Create the splits
agn_train     = Subset(training_dataset, full_train_subset)
agn_unlabeled = Subset(training_dataset, full_unlabeled_subset)
testing_dataset = Subset(testing_dataset, full_test_subset)

label_count = {0:0, 1:0, 2:0, 3:0}
for i in range(len(agn_train)):
    the_label = agn_train[i]["labels"].item()
    label_count[the_label] += 1
print(label_count)

label_count = label_count = {0:0, 1:0, 2:0, 3:0}
for i in range(len(testing_dataset)):
    the_label = testing_dataset[i]["labels"].item()
    label_count[the_label] += 1
print(label_count)

## Preparing the Model

The model here is a CustomBERTModel based on https://arxiv.org/abs/1810.04805 and https://arxiv.org/abs/1908.08962. It uses a bidirectional tranformer scheme that has been pretrained based on masked language modeling and next sentence prediction. The model we use is a tailored version of BERT-small from HuggingFace's transformers library that can be found here: https://huggingface.co/prajjwal1/bert-small. The additions allow for sentiment classification and for usage of DISTIL's strategies.

In [ ]:
from transformers import BertModel

class CustomBERTModel(nn.Module):
    def __init__(self, config):
        super(CustomBERTModel, self).__init__()
        self.config = config
        self.bert = BertModel.from_pretrained("prajjwal1/bert-small")
        classifier_dropout = (config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob)
        self.dropout = nn.Dropout(classifier_dropout)
        self.linear1 = nn.Linear(512,config.num_classes)
        
    def get_embedding_dim(self):
        return 512

    def forward(self, input_ids=None,
               attention_mask=None,
               token_type_ids=None,
               position_ids=None,
               head_mask=None,
               inputs_embeds=None,
               labels=None,
               output_attentions=None,
               output_hidden_states=None,
               return_dict=None, freeze=False, last=False):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        if freeze:
            with torch.no_grad():
                outputs = self.bert(input_ids,
                                    attention_mask=attention_mask,
                                    token_type_ids=token_type_ids,
                                    position_ids=position_ids,
                                    head_mask=head_mask,
                                    inputs_embeds=inputs_embeds,
                                    output_attentions=output_attentions,
                                    output_hidden_states=output_hidden_states,
                                    return_dict=return_dict)
                pooled_output = self.dropout(outputs["pooler_output"])
        else:
            outputs = self.bert(input_ids,
                                attention_mask=attention_mask,
                                token_type_ids=token_type_ids,
                                position_ids=position_ids,
                                head_mask=head_mask,
                                inputs_embeds=inputs_embeds,
                                output_attentions=output_attentions,
                                output_hidden_states=output_hidden_states,
                                return_dict=return_dict)
            pooled_output = self.dropout(outputs["pooler_output"])
        linear1_output = self.linear1(pooled_output)
        if last:
            return linear1_output, pooled_output
        else:
            return linear1_output

base_dir = "models"
os.makedirs(base_dir, exist_ok = True)
model_directory = os.path.join(base_dir, 'base_model.pth')

model = CustomBERTModel(configuration)
if(torch.cuda.is_available()):
    device = "cuda"
else:
    device = "cpu"
model.to(device)

CustomBERTModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise

## data_train Class

Here, we define a training procedure that will help us fine-tune the BERT model. Specifically, this data_train class is modeled after the data_train class already included in distil/distil/active_learning_stratgies. However, the following data_train class is designed to work with the dataset format of twitter_eval and the CustomBERTModel.

In [ ]:
#custom training
class data_train:
    
    def __init__(self, training_dataset, net, args):

        self.training_dataset = training_dataset
        self.net = net
        self.args = args
        
        self.n_pool = len(training_dataset)
        
        if 'islogs' not in args:
            self.args['islogs'] = False

        if 'optimizer' not in args:
            self.args['optimizer'] = 'adam'
        
        if 'isverbose' not in args:
            self.args['isverbose'] = False
        
        if 'isreset' not in args:
            self.args['isreset'] = True

        if 'n_epoch' not in args:
            self.args['n_epoch'] = 200
            
        if 'criterion' not in args:
            self.args['criterion'] = nn.CrossEntropyLoss()
            
        if 'device' not in args:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = args['device']

    def update_index(self, idxs_lb):
        self.idxs_lb = idxs_lb

    def update_data(self, new_training_dataset):
        """
        Updates the training dataset with the provided new training dataset
        
        Parameters
        ----------
        new_training_dataset: torch.utils.data.Dataset
            The new training dataset
        """
        self.training_dataset = new_training_dataset

    def get_acc_on_set(self, test_dataset):
        
        """
        Calculates and returns the accuracy on the given dataset to test
        
        Parameters
        ----------
        test_dataset: torch.utils.data.Dataset
            The dataset to test
        Returns
        -------
        accFinal: float
            The fraction of data points whose predictions by the current model match their targets
        """	
        
        try:
            self.clf
        except:
            self.clf = self.net

        if test_dataset is None:
            raise ValueError("Test data not present")
        
        if 'batch_size' in self.args:
            batch_size = self.args['batch_size']
        else:
            batch_size = 1 
 
        loader_te = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
        self.clf.eval()
        accFinal = 0.

        self.clf = self.clf.to(device=self.device)
        for batch in loader_te:  
            batch = {k: v.to(self.device) for k, v in batch.items()}
            with torch.no_grad():
              out = self.clf(**batch)
              accFinal += torch.sum(1.0*(torch.max(out,1)[1] == batch['labels'].view(-1))).item()

        return accFinal / len(test_dataset)

    def _train(self, epoch, loader_tr, optimizer):
        self.clf.train()
        accFinal = 0.
        criterion = self.args['criterion']
        criterion.reduction = "mean"

        for batch in loader_tr:
            batch = {k: v.to(self.device) for k, v in batch.items()}
            out = self.clf(**batch, freeze=True)
            loss = criterion(out, batch['labels'].view(-1))
            accFinal += torch.sum((torch.max(out,1)[1] == batch['labels'].view(-1)).float()).item()
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        return accFinal / len(loader_tr.dataset), loss

    def train(self, gradient_weights=None):

        print('Training..')
        train_logs = []
        
        try:
            self.clf
        except:
            self.clf = self.net
        
        if self.args['isreset']:
            self.clf.linear1.reset_parameters()
        
        if 'batch_size' in self.args:
            batch_size = self.args['batch_size']
        else:
            batch_size = 1

        loader_tr = DataLoader(self.training_dataset, shuffle=True, batch_size=batch_size)
        epoch = 1
        accCurrent = 0

        if self.args['optimizer'] == 'adam':
            optimizer = AdamW(self.clf.parameters(), lr = self.args['lr'])

        while epoch < self.args['n_epoch']: 
            self.clf = self.clf.to(device=self.device)
            accCurrent, lossCurrent = self._train(epoch, loader_tr, optimizer)
            
            epoch += 1
            if epoch % 5 == 0:
                print(str(epoch) + ' training accuracy: ' + str(accCurrent), flush=True)

            log_string = 'Epoch:' + str(epoch) + '- training accuracy:'+str(accCurrent)+'- training loss:'+str(lossCurrent)
            train_logs.append(log_string)

        print('Epoch:', str(epoch), 'Training accuracy:', round(accCurrent, 3), flush=True)

        if self.args['islogs']:
            return self.clf, train_logs
        else:
            return self.clf

## Training an Initial Model

Now that our model, labeled data, and unlabeled data are prepared, we can now train an initial model. Here, we use AdamW with a learning rate of 1e-4 and batch size of 5 for 200 epochs.

In [ ]:
args = {'lr':1e-4, 'batch_size':5, 'n_epoch': 200, 'optimizer':'adam'} 
dt = data_train(agn_train, model, args)
clf = dt.train()
torch.save(clf.state_dict(), model_directory)

model = CustomBERTModel(configuration)
state_dict = torch.load(model_directory)
model.load_state_dict(state_dict)

Training..
5 training accuracy: 0.6825
10 training accuracy: 0.78875
15 training accuracy: 0.82125
20 training accuracy: 0.83125
25 training accuracy: 0.82875
30 training accuracy: 0.84875
35 training accuracy: 0.84125
40 training accuracy: 0.85625
45 training accuracy: 0.85625
50 training accuracy: 0.865
55 training accuracy: 0.86125
60 training accuracy: 0.85875
65 training accuracy: 0.85625
70 training accuracy: 0.8575
75 training accuracy: 0.85625
80 training accuracy: 0.8625
85 training accuracy: 0.8675
90 training accuracy: 0.8675
95 training accuracy: 0.86375
100 training accuracy: 0.86375
105 training accuracy: 0.8675
110 training accuracy: 0.875
115 training accuracy: 0.86625
120 training accuracy: 0.87125
125 training accuracy: 0.875
130 training accuracy: 0.89
135 training accuracy: 0.88875
140 training accuracy: 0.875
145 training accuracy: 0.87625
150 training accuracy: 0.87875
155 training accuracy: 0.875
160 training accuracy: 0.86875
165 training accuracy: 0.87625
170 t

/tmp/ipykernel_62931/1089867559.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_directory)


<All keys matched successfully>

## Active Learning Strategies

Now that we have the initial model, we can use one of DISTIL's active learning strategies to select examples from twitter_eval to label and to use in future training.

### Random Sampling
This strategy is often used as a baseline, where we pick a subset of unlabeled points randomly. Here, we create a instance of `distil.active_learning_strategies.random_sampling.RandomSampling` by passing following parameters:

**training_dataset** – The labeled dataset, which has a PyTorch interface that returns a tensor of data and a tensor of labels OR a dictionary with a "labels" key.

**unlabeled_dataset** – The unlabeled dataset, which has a PyTorch interface that returns a tensor of data OR a dictionary.

**net** – Model architecture used for training. Could be instance of models defined in distil.utils.models or something similar (in this instance, a Huggingface transformer).

**nclasses** – No. of classes in the dataset

**args**– This dictionary should have ‘batch_size’ as a key. 'batch_size' should be such that one can exploit the benefits of tensorization while honouring the resourse constraits. This ‘batch_size’ therefore can be different than the one used for training.


In [ ]:

from copy import deepcopy

initial_agn_train = deepcopy(agn_train)
initial_unlabeled = deepcopy(agn_unlabeled)
# Loop over each class
for target_class in range(nclasses):
    print(f"\n========= Class {target_class} =========")

    # Restrict test set to only this class
    class_test_indices = [i for i in range(len(testing_dataset)) if testing_dataset[i]['labels'].item() == target_class]
    class_test_set = Subset(testing_dataset, class_test_indices)

    # Reinitialize and load the pretrained model
   
    model = CustomBERTModel(configuration)
    state_dict = torch.load(model_directory)
    model.load_state_dict(state_dict, strict=False)

    # Set up training args again
    args = {'lr':1e-4, 'batch_size':5, 'n_epoch': 200, 'optimizer':'adam'}
    agn_train = deepcopy(initial_agn_train)
    dt = data_train(agn_train, model, args)

    # Create AL strategy instance
    strategy_args = {'batch_size' : 5}
    agn_unlabeled = deepcopy(initial_unlabeled)
    strategy = RandomSampling(agn_train, LabeledToUnlabeledDataset(agn_unlabeled), model, nclasses, strategy_args)
    strategy.update_model(model)

    # Track accuracy across rounds
    acc = np.zeros(n_rounds)
    acc[0] = dt.get_acc_on_set(class_test_set)
    print(f"Initial test accuracy on class {target_class}: {round(acc[0]*100, 2)}%")

    # Active Learning Loop
    for rd in range(1, n_rounds):
        print('-------------------------------------------------')
        print(f'Round {rd} for class {target_class}')
        print('-------------------------------------------------')

        # Disable augmentation for selection
        idx = strategy.select(budget)

        # Add selected samples to training set
        selected_subset = Subset(agn_unlabeled, idx)
        agn_train = ConcatDataset([agn_train, selected_subset])

        # Update unlabeled pool
        remaining_indices = list(set(range(len(agn_unlabeled))) - set(idx))
        agn_unlabeled = Subset(agn_unlabeled, remaining_indices)

        print("Number of training samples:", len(agn_train))

        # Update strategy and training data
        strategy.update_data(agn_train, LabeledToUnlabeledDataset(agn_unlabeled))
        dt.update_data(agn_train)

        # Retrain model
        clf = dt.train()
        strategy.update_model(clf)

        # Evaluate on current class only
        acc[rd] = dt.get_acc_on_set(class_test_set)
        print(f"Accuracy on class {target_class}: {round(acc[rd]*100, 2)}%")

    print(f"Training completed for class {target_class}")

    # Save accuracy results
    dir = os.path.join('results', 'AGN', 'baseline')
    os.makedirs(dir, exist_ok=True)
    save_path = os.path.join(dir, f'class_{target_class}_random.txt')
    with open(save_path, 'w') as f:
        for a in acc:
            f.write(f"{a}\n")
    print(f"Saved class-wise accuracy to: {save_path}")


========= Class 0 =========


/tmp/ipykernel_62931/746307775.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_directory)


Initial test accuracy on class 0: 85.1%
-------------------------------------------------
Round 1 for class 0
-------------------------------------------------
Number of training samples: 1100
Training..
5 training accuracy: 0.740909090909091


10 training accuracy: 0.8163636363636364
15 training accuracy: 0.8354545454545454
20 training accuracy: 0.8363636363636363
25 training accuracy: 0.8590909090909091
30 training accuracy: 0.8527272727272728
35 training accuracy: 0.8472727272727273
40 training accuracy: 0.8554545454545455
45 training accuracy: 0.8563636363636363
50 training accuracy: 0.8618181818181818
55 training accuracy: 0.8563636363636363
60 training accuracy: 0.8681818181818182
65 training accuracy: 0.8645454545454545
70 training accuracy: 0.8681818181818182
75 training accuracy: 0.8727272727272727
80 training accuracy: 0.860909090909091
85 training accuracy: 0.8690909090909091
90 training accuracy: 0.8718181818181818
95 training accuracy: 0.8672727272727273
100 training accuracy: 0.8663636363636363
105 training accuracy: 0.8781818181818182
110 training accuracy: 0.8709090909090909
115 training accuracy: 0.8736363636363637
120 training accuracy: 0.8763636363636363
125 training accuracy: 0.8709090909090909
130 trainin

### Entropy Sampling
A very basic strategy to select unlabeled points is entropy sampling, where we select samples about which the model is most uncertain by measuring the entropy of the class prediction. Hence, a valid strategy is to select those points in the unlabeled set with highest entropy (maximum uncertainty). Specifically, let $z = f(x)$ be the output from the model with class $i$ having score $z_i$. By applying a softmax, we obtain probabilities that we can use: 

$$\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$ 

Using the softmax probabilities, entropy sampling selects a subset based on the following objective:

 $$\text{argmin}_{A \subseteq U, |A| = k} \sum_{x \in A} \left[ -\sum_j \sigma(z)_j*\log(\sigma(z)_j) \right]$$

Here, we create a instance of `distil.active_learning_strategies.entropy_sampling.EntropySampling` with the same parameters passed to `distil.active_learning_strategies.random_sampling.RandomSampling`.

**Reload Data**

Here we reload, reprocess, and retokenize the data and prepare for a different active learning strategy.

In [ ]:


dataset = load_dataset("fancyzhx/ag_news")

train_dataset = dataset['train']
test_dataset = dataset['test']

# Text preprocessing functions
def to_lower(ex):
    ex['text'] = ex['text'].lower()
    return ex

def remove_punc(ex):
    ex['text'] = ''.join(x for x in ex['text'] if x not in string.punctuation)
    return ex

def to_ascii(ex):
    ex['text'] = ''.join(x for x in unicodedata.normalize('NFD', ex['text']) if unicodedata.category(x) != 'Mn')
    return ex

stopwords = [ "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at", "be", "because", 
             "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down", "during",
             "each", "few", "for", "from", "further", "had", "has", "have", "having", "he", "he'd", "he'll", "he's", "her", "here", 
             "here's", "hers", "herself", "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into",
             "is", "it", "it's", "its", "itself", "let's", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or",
             "other", "ought", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "she'd", "she'll", "she's", "should", 
             "so", "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves", "then", "there", "there's",
             "these", "they", "they'd", "they'll", "they're", "they've", "this", "those", "through", "to", "too", "under", "until", "up",
             "very", "was", "we", "we'd", "we'll", "we're", "we've", "were", "what", "what's", "when", "when's", "where", "where's",
             "which", "while", "who", "who's", "whom", "why", "why's", "with", "would", "you", "you'd", "you'll", "you're", "you've",
             "your", "yours", "yourself", "yourselves" ]

def remove_stopwords(ex):
    ex['text'] = ' '.join(x for x in ex['text'].split() if x not in stopwords)
    return ex

def process_data(data):
    data1 = data.map(remove_punc)
    data2 = data1.map(to_ascii)
    data3 = data2.map(to_lower)
    data4 = data3.map(remove_stopwords)
    return data4

# Apply processing
training_data = process_data(train_dataset)
testing_data = process_data(test_dataset)


from transformers import AutoTokenizer, get_scheduler, BertConfig, AdamW, BertModel

configuration = BertConfig()
setattr(configuration, 'l1', 512)
setattr(configuration, 'num_classes', 4)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_func(ex):
    return tokenizer(ex['text'], padding='max_length', truncation=True)

# Removes unnecessary columns and renames columns to match the model specification
train1 = training_data.map(tokenize_func, batched=True)
test1 = testing_data.map(tokenize_func, batched=True)
train2 = train1.remove_columns("text")
test2 = test1.remove_columns("text")
training_dataset = train2.rename_column("label", "labels")
testing_dataset = test2.rename_column("label", "labels")

training_dataset.set_format("torch")
testing_dataset.set_format("torch")

# Define the number of active learning rounds to conduct, the budget, and the number of classes in AGN dataset
nclasses = 4
n_rounds = 6
budget = 300

train_size_pc     = 200
test_size_pc      = 1000
unlabeled_size_pc = 5000

# Ensure reproducibility
np.random.seed(42)

# Generate random index lists for subset creation
full_train_subset = []
full_unlabeled_subset = []
full_test_subset = []

for class_num in range(nclasses):
    train_idx_pool = []
    for train_idx in range(len(training_dataset)):
        if training_dataset[train_idx]["labels"].item() == class_num:
            train_idx_pool.append(train_idx)
    test_idx_pool = []
    for test_idx in range(len(testing_dataset)):
        if testing_dataset[test_idx]["labels"].item() == class_num:
            test_idx_pool.append(test_idx)

    # Get a random subset from each pool
    print(f"Class {class_num} - Training pool size: {len(train_idx_pool)}, Testing pool size: {len(test_idx_pool)}")
    train_idx_subset = np.random.choice(train_idx_pool, size=train_size_pc, replace = False).tolist()
    remaining_idx = list(set(train_idx_pool) - set(train_idx_subset))
    unlabeled_idx_subset = np.random.choice(remaining_idx, size=unlabeled_size_pc, replace = False).tolist()
    test_idx_subset = np.random.choice(test_idx_pool, size=test_size_pc, replace = False).tolist()
    full_train_subset.extend(train_idx_subset)
    full_unlabeled_subset.extend(unlabeled_idx_subset)
    full_test_subset.extend(test_idx_subset)


# Create the splits
agn_train     = Subset(training_dataset, full_train_subset)
agn_unlabeled = Subset(training_dataset, full_unlabeled_subset)
testing_dataset = Subset(testing_dataset, full_test_subset)

model = CustomBERTModel(configuration)
state_dict = torch.load(model_directory)
model.load_state_dict(state_dict)

2025-06-25 01:35:55.934671: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-25 01:35:55.981059: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-25 01:35:56.836312: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Class 0 - Training pool size: 30000, Testing pool size: 1900
Class 1 - Training pool size: 30000, Testing pool size: 1900
Class 2 - Training pool size: 30000, Testing pool size: 1900
Class 3 - Training pool size: 30000, Testing pool size: 1900


NameError: name 'CustomBERTModel' is not defined

In [ ]:

from copy import deepcopy
from distil.utils.utils import LabeledToUnlabeledDataset  

initial_agn_train = deepcopy(agn_train)
initial_unlabeled = deepcopy(agn_unlabeled)
# Loop over each class
for target_class in range(nclasses):
    print(f"\n========= Class {target_class} =========")

    # Restrict test set to only this class
    class_test_indices = [i for i in range(len(testing_dataset)) if testing_dataset[i]['labels'].item() == target_class]
    class_test_set = Subset(testing_dataset, class_test_indices)

    # Reinitialize and load the pretrained model
   
    model = CustomBERTModel(configuration)
    state_dict = torch.load(model_directory)
    model.load_state_dict(state_dict, strict=False)

    # Set up training args again
    args = {'lr':1e-4, 'batch_size':5, 'n_epoch': 200, 'optimizer':'adam'}
    agn_train = deepcopy(initial_agn_train)
    dt = data_train(agn_train, model, args)

    # Create AL strategy instance
    strategy_args = {'batch_size' : 5}
    agn_unlabeled = deepcopy(initial_unlabeled)
    strategy = EntropySampling(agn_train, LabeledToUnlabeledDataset(agn_unlabeled), model, nclasses, strategy_args)
    strategy.update_model(model)

    # Track accuracy across rounds
    acc = np.zeros(n_rounds)
    acc[0] = dt.get_acc_on_set(class_test_set)
    print(f"Initial test accuracy on class {target_class}: {round(acc[0]*100, 2)}%")

    # Active Learning Loop
    for rd in range(1, n_rounds):
        print('-------------------------------------------------')
        print(f'Round {rd} for class {target_class}')
        print('-------------------------------------------------')

        # Disable augmentation for selection
        idx = strategy.select(budget)

        # Add selected samples to training set
        selected_subset = Subset(agn_unlabeled, idx)
        agn_train = ConcatDataset([agn_train, selected_subset])

        # Update unlabeled pool
        remaining_indices = list(set(range(len(agn_unlabeled))) - set(idx))
        agn_unlabeled = Subset(agn_unlabeled, remaining_indices)

        print("Number of training samples:", len(agn_train))

        # Update strategy and training data
        strategy.update_data(agn_train, LabeledToUnlabeledDataset(agn_unlabeled))
        dt.update_data(agn_train)

        # Retrain model
        clf = dt.train()
        strategy.update_model(clf)

        # Evaluate on current class only
        acc[rd] = dt.get_acc_on_set(class_test_set)
        print(f"Accuracy on class {target_class}: {round(acc[rd]*100, 2)}%")

    print(f"Training completed for class {target_class}")

    # Save accuracy results
    dir = os.path.join('results', 'AGN', 'baseline')
    os.makedirs(dir, exist_ok=True)
    save_path = os.path.join(dir, f'class_{target_class}_entropy.txt')
    with open(save_path, 'w') as f:
        for a in acc:
            f.write(f"{a}\n")
    print(f"Saved class-wise accuracy to: {save_path}")

NameError: name 'agn_train' is not defined

### BADGE
This method is based on the paper [Deep Batch Active Learning by Diverse, Uncertain Gradient Lower Bounds](https://arxiv.org/abs/1906.03671). The strategy is meant to select points that are both diverse (e.g., their embeddings span multiple directions) and uncertain (e.g., their contribution to the loss is large). The following steps are taken:

* Calculate the pseudo-label for each point in the unlabeled set. The pseudo-label is the class with the highest probability.
* Compute the cross-entropy loss for each point in the unlabeled set using this pseudo-label.
* Obtain the resulting loss gradients on the last linear layer of the model for each point. (These are referred to as the hypothesized loss gradients.)
* Using these gradients as a form of embedding for each unlabeled point, run k-means++ initialization on this embedding set, retrieving $k$ centers. Each center is a point from the unlabeled set, and $k$ represents the active learning budget.
* Request labels for the $k$ points whose embeddings were selected.

Here we create a instance of `distil.active_learning_strategies.badge.BADGE` with same parameters passed to `distil.active_learning_strategies.random_sampling.RandomSampling`.

**Reload Data**

Here we reload, reprocess, and retokenize the data and prepare for a different active learning strategy.

In [ ]:


dataset = load_dataset("fancyzhx/ag_news")

train_dataset = dataset['train']
test_dataset = dataset['test']

# Text preprocessing functions
def to_lower(ex):
    ex['text'] = ex['text'].lower()
    return ex

def remove_punc(ex):
    ex['text'] = ''.join(x for x in ex['text'] if x not in string.punctuation)
    return ex

def to_ascii(ex):
    ex['text'] = ''.join(x for x in unicodedata.normalize('NFD', ex['text']) if unicodedata.category(x) != 'Mn')
    return ex

stopwords = [ "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at", "be", "because", 
             "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down", "during",
             "each", "few", "for", "from", "further", "had", "has", "have", "having", "he", "he'd", "he'll", "he's", "her", "here", 
             "here's", "hers", "herself", "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into",
             "is", "it", "it's", "its", "itself", "let's", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or",
             "other", "ought", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "she'd", "she'll", "she's", "should", 
             "so", "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves", "then", "there", "there's",
             "these", "they", "they'd", "they'll", "they're", "they've", "this", "those", "through", "to", "too", "under", "until", "up",
             "very", "was", "we", "we'd", "we'll", "we're", "we've", "were", "what", "what's", "when", "when's", "where", "where's",
             "which", "while", "who", "who's", "whom", "why", "why's", "with", "would", "you", "you'd", "you'll", "you're", "you've",
             "your", "yours", "yourself", "yourselves" ]

def remove_stopwords(ex):
    ex['text'] = ' '.join(x for x in ex['text'].split() if x not in stopwords)
    return ex

def process_data(data):
    data1 = data.map(remove_punc)
    data2 = data1.map(to_ascii)
    data3 = data2.map(to_lower)
    data4 = data3.map(remove_stopwords)
    return data4

# Apply processing
training_data = process_data(train_dataset)
testing_data = process_data(test_dataset)


from transformers import AutoTokenizer, get_scheduler, BertConfig, AdamW, BertModel

configuration = BertConfig()
setattr(configuration, 'l1', 512)
setattr(configuration, 'num_classes', 4)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_func(ex):
    return tokenizer(ex['text'], padding='max_length', truncation=True)

# Removes unnecessary columns and renames columns to match the model specification
train1 = training_data.map(tokenize_func, batched=True)
test1 = testing_data.map(tokenize_func, batched=True)
train2 = train1.remove_columns("text")
test2 = test1.remove_columns("text")
training_dataset = train2.rename_column("label", "labels")
testing_dataset = test2.rename_column("label", "labels")

training_dataset.set_format("torch")
testing_dataset.set_format("torch")

# Define the number of active learning rounds to conduct, the budget, and the number of classes in AGN dataset
nclasses = 4
n_rounds = 6
budget = 300

train_size_pc     = 200
test_size_pc      = 1000
unlabeled_size_pc = 5000

# Ensure reproducibility
np.random.seed(42)

# Generate random index lists for subset creation
full_train_subset = []
full_unlabeled_subset = []
full_test_subset = []

for class_num in range(nclasses):
    train_idx_pool = []
    for train_idx in range(len(training_dataset)):
        if training_dataset[train_idx]["labels"].item() == class_num:
            train_idx_pool.append(train_idx)
    test_idx_pool = []
    for test_idx in range(len(testing_dataset)):
        if testing_dataset[test_idx]["labels"].item() == class_num:
            test_idx_pool.append(test_idx)

    # Get a random subset from each pool
    print(f"Class {class_num} - Training pool size: {len(train_idx_pool)}, Testing pool size: {len(test_idx_pool)}")
    train_idx_subset = np.random.choice(train_idx_pool, size=train_size_pc, replace = False).tolist()
    remaining_idx = list(set(train_idx_pool) - set(train_idx_subset))
    unlabeled_idx_subset = np.random.choice(remaining_idx, size=unlabeled_size_pc, replace = False).tolist()
    test_idx_subset = np.random.choice(test_idx_pool, size=test_size_pc, replace = False).tolist()
    full_train_subset.extend(train_idx_subset)
    full_unlabeled_subset.extend(unlabeled_idx_subset)
    full_test_subset.extend(test_idx_subset)


# Create the splits
agn_train     = Subset(training_dataset, full_train_subset)
agn_unlabeled = Subset(training_dataset, full_unlabeled_subset)
testing_dataset = Subset(testing_dataset, full_test_subset)

model = CustomBERTModel(configuration)
state_dict = torch.load(model_directory)
model.load_state_dict(state_dict)

Some weights of the model checkpoint at prajjwal1/bert-small were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


<All keys matched successfully>

In [ ]:

from copy import deepcopy

initial_agn_train = deepcopy(agn_train)
initial_unlabeled = deepcopy(agn_unlabeled)
# Loop over each class
for target_class in range(nclasses):
    print(f"\n========= Class {target_class} =========")

    # Restrict test set to only this class
    class_test_indices = [i for i in range(len(testing_dataset)) if testing_dataset[i]['labels'].item() == target_class]
    class_test_set = Subset(testing_dataset, class_test_indices)

    # Reinitialize and load the pretrained model
   
    model = CustomBERTModel(configuration)
    state_dict = torch.load(model_directory)
    model.load_state_dict(state_dict, strict=False)

    # Set up training args again
    args = {'lr':1e-4, 'batch_size':5, 'n_epoch': 200, 'optimizer':'adam'}
    agn_train = deepcopy(initial_agn_train)
    dt = data_train(agn_train, model, args)

    # Create AL strategy instance
    strategy_args = {'batch_size' : 5}
    agn_unlabeled = deepcopy(initial_unlabeled)
    strategy = BADGE(agn_train, LabeledToUnlabeledDataset(agn_unlabeled), model, nclasses, strategy_args)
    strategy.update_model(model)

    # Track accuracy across rounds
    acc = np.zeros(n_rounds)
    acc[0] = dt.get_acc_on_set(class_test_set)
    print(f"Initial test accuracy on class {target_class}: {round(acc[0]*100, 2)}%")

    # Active Learning Loop
    for rd in range(1, n_rounds):
        print('-------------------------------------------------')
        print(f'Round {rd} for class {target_class}')
        print('-------------------------------------------------')

        # Disable augmentation for selection
        idx = strategy.select(budget)

        # Add selected samples to training set
        selected_subset = Subset(agn_unlabeled, idx)
        agn_train = ConcatDataset([agn_train, selected_subset])

        # Update unlabeled pool
        remaining_indices = list(set(range(len(agn_unlabeled))) - set(idx))
        agn_unlabeled = Subset(agn_unlabeled, remaining_indices)

        print("Number of training samples:", len(agn_train))

        # Update strategy and training data
        strategy.update_data(agn_train, LabeledToUnlabeledDataset(agn_unlabeled))
        dt.update_data(agn_train)

        # Retrain model
        clf = dt.train()
        strategy.update_model(clf)

        # Evaluate on current class only
        acc[rd] = dt.get_acc_on_set(class_test_set)
        print(f"Accuracy on class {target_class}: {round(acc[rd]*100, 2)}%")

    print(f"Training completed for class {target_class}")

    # Save accuracy results
    dir = os.path.join('results', 'AGN', 'baseline')
    os.makedirs(dir, exist_ok=True)
    save_path = os.path.join(dir, f'class_{target_class}_badge.txt')
    with open(save_path, 'w') as f:
        for a in acc:
            f.write(f"{a}\n")
    print(f"Saved class-wise accuracy to: {save_path}")


========= Class 0 =========


/tmp/ipykernel_55753/117392792.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_directory)


Initial test accuracy on class 0: 83.4%
-------------------------------------------------
Round 1 for class 0
-------------------------------------------------


ValueError: too many values to unpack (expected 2)

### Least Confidence Sampling

A very basic strategy to select unlabeled points is least confidence sampling, where we select samples about which the model is most uncertain by measuring the confidence of the class prediction. Hence, a valid strategy is to select those points in the unlabeled set with the least confident predictions (lowest maximum probability score). Specifically, let $z = f(x)$ be the output from the model with class $i$ having score $z_i$. By applying a softmax, we obtain probabilities that we can use: 

$$\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$ 

Using the softmax probabilities, entropy sampling selects a subset based on the following objective:

 $$\text{argmin}_{A \subseteq U, |A| = k} \sum_{x \in A} \left[ \max_j \sigma(z)_j \right]$$

Here, we create a instance of `distil.active_learning_strategies.entropy_sampling.LeastConfidenceSampling` with the same parameters passed to `distil.active_learning_strategies.random_sampling.RandomSampling`.

**Reload Data**

Here we reload, reprocess, and retokenize the data and prepare for a different active learning strategy.

In [ ]:


dataset = load_dataset("fancyzhx/ag_news")

train_dataset = dataset['train']
test_dataset = dataset['test']

# Text preprocessing functions
def to_lower(ex):
    ex['text'] = ex['text'].lower()
    return ex

def remove_punc(ex):
    ex['text'] = ''.join(x for x in ex['text'] if x not in string.punctuation)
    return ex

def to_ascii(ex):
    ex['text'] = ''.join(x for x in unicodedata.normalize('NFD', ex['text']) if unicodedata.category(x) != 'Mn')
    return ex

stopwords = [ "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at", "be", "because", 
             "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down", "during",
             "each", "few", "for", "from", "further", "had", "has", "have", "having", "he", "he'd", "he'll", "he's", "her", "here", 
             "here's", "hers", "herself", "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into",
             "is", "it", "it's", "its", "itself", "let's", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or",
             "other", "ought", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "she'd", "she'll", "she's", "should", 
             "so", "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves", "then", "there", "there's",
             "these", "they", "they'd", "they'll", "they're", "they've", "this", "those", "through", "to", "too", "under", "until", "up",
             "very", "was", "we", "we'd", "we'll", "we're", "we've", "were", "what", "what's", "when", "when's", "where", "where's",
             "which", "while", "who", "who's", "whom", "why", "why's", "with", "would", "you", "you'd", "you'll", "you're", "you've",
             "your", "yours", "yourself", "yourselves" ]

def remove_stopwords(ex):
    ex['text'] = ' '.join(x for x in ex['text'].split() if x not in stopwords)
    return ex

def process_data(data):
    data1 = data.map(remove_punc)
    data2 = data1.map(to_ascii)
    data3 = data2.map(to_lower)
    data4 = data3.map(remove_stopwords)
    return data4

# Apply processing
training_data = process_data(train_dataset)
testing_data = process_data(test_dataset)


from transformers import AutoTokenizer, get_scheduler, BertConfig, AdamW, BertModel

configuration = BertConfig()
setattr(configuration, 'l1', 512)
setattr(configuration, 'num_classes', 4)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_func(ex):
    return tokenizer(ex['text'], padding='max_length', truncation=True)

# Removes unnecessary columns and renames columns to match the model specification
train1 = training_data.map(tokenize_func, batched=True)
test1 = testing_data.map(tokenize_func, batched=True)
train2 = train1.remove_columns("text")
test2 = test1.remove_columns("text")
training_dataset = train2.rename_column("label", "labels")
testing_dataset = test2.rename_column("label", "labels")

training_dataset.set_format("torch")
testing_dataset.set_format("torch")

# Define the number of active learning rounds to conduct, the budget, and the number of classes in AGN dataset
nclasses = 4
n_rounds = 6
budget = 300

train_size_pc     = 200
test_size_pc      = 1000
unlabeled_size_pc = 5000

# Ensure reproducibility
np.random.seed(42)

# Generate random index lists for subset creation
full_train_subset = []
full_unlabeled_subset = []
full_test_subset = []

for class_num in range(nclasses):
    train_idx_pool = []
    for train_idx in range(len(training_dataset)):
        if training_dataset[train_idx]["labels"].item() == class_num:
            train_idx_pool.append(train_idx)
    test_idx_pool = []
    for test_idx in range(len(testing_dataset)):
        if testing_dataset[test_idx]["labels"].item() == class_num:
            test_idx_pool.append(test_idx)

    # Get a random subset from each pool
    print(f"Class {class_num} - Training pool size: {len(train_idx_pool)}, Testing pool size: {len(test_idx_pool)}")
    train_idx_subset = np.random.choice(train_idx_pool, size=train_size_pc, replace = False).tolist()
    remaining_idx = list(set(train_idx_pool) - set(train_idx_subset))
    unlabeled_idx_subset = np.random.choice(remaining_idx, size=unlabeled_size_pc, replace = False).tolist()
    test_idx_subset = np.random.choice(test_idx_pool, size=test_size_pc, replace = False).tolist()
    full_train_subset.extend(train_idx_subset)
    full_unlabeled_subset.extend(unlabeled_idx_subset)
    full_test_subset.extend(test_idx_subset)


# Create the splits
agn_train     = Subset(training_dataset, full_train_subset)
agn_unlabeled = Subset(training_dataset, full_unlabeled_subset)
testing_dataset = Subset(testing_dataset, full_test_subset)

model = CustomBERTModel(configuration)
state_dict = torch.load(model_directory)
model.load_state_dict(state_dict)

Some weights of the model checkpoint at prajjwal1/bert-small were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


<All keys matched successfully>

In [ ]:

from copy import deepcopy

initial_agn_train = deepcopy(agn_train)
initial_unlabeled = deepcopy(agn_unlabeled)
# Loop over each class
for target_class in range(nclasses):
    print(f"\n========= Class {target_class} =========")

    # Restrict test set to only this class
    class_test_indices = [i for i in range(len(testing_dataset)) if testing_dataset[i]['labels'].item() == target_class]
    class_test_set = Subset(testing_dataset, class_test_indices)

    # Reinitialize and load the pretrained model
   
    model = CustomBERTModel(configuration)
    state_dict = torch.load(model_directory)
    model.load_state_dict(state_dict, strict=False)

    # Set up training args again
    args = {'lr':1e-4, 'batch_size':5, 'n_epoch': 200, 'optimizer':'adam'}
    agn_train = deepcopy(initial_agn_train)
    dt = data_train(agn_train, model, args)

    # Create AL strategy instance
    strategy_args = {'batch_size' : 5}
    agn_unlabeled = deepcopy(initial_unlabeled)
    strategy = LeastConfidenceSampling(agn_train, LabeledToUnlabeledDataset(agn_unlabeled), model, nclasses, strategy_args)
    strategy.update_model(model)

    # Track accuracy across rounds
    acc = np.zeros(n_rounds)
    acc[0] = dt.get_acc_on_set(class_test_set)
    print(f"Initial test accuracy on class {target_class}: {round(acc[0]*100, 2)}%")

    # Active Learning Loop
    for rd in range(1, n_rounds):
        print('-------------------------------------------------')
        print(f'Round {rd} for class {target_class}')
        print('-------------------------------------------------')

        # Disable augmentation for selection
        idx = strategy.select(budget)

        # Add selected samples to training set
        selected_subset = Subset(agn_unlabeled, idx)
        agn_train = ConcatDataset([agn_train, selected_subset])

        # Update unlabeled pool
        remaining_indices = list(set(range(len(agn_unlabeled))) - set(idx))
        agn_unlabeled = Subset(agn_unlabeled, remaining_indices)

        print("Number of training samples:", len(agn_train))

        # Update strategy and training data
        strategy.update_data(agn_train, LabeledToUnlabeledDataset(agn_unlabeled))
        dt.update_data(agn_train)

        # Retrain model
        clf = dt.train()
        strategy.update_model(clf)

        # Evaluate on current class only
        acc[rd] = dt.get_acc_on_set(class_test_set)
        print(f"Accuracy on class {target_class}: {round(acc[rd]*100, 2)}%")

    print(f"Training completed for class {target_class}")

    # Save accuracy results
    dir = os.path.join('results', 'AGN', 'baseline')
    os.makedirs(dir, exist_ok=True)
    save_path = os.path.join(dir, f'class_{target_class}_leastConfidence.txt')
    with open(save_path, 'w') as f:
        for a in acc:
            f.write(f"{a}\n")
    print(f"Saved class-wise accuracy to: {save_path}")


========= Class 0 =========


/tmp/ipykernel_55753/2300914948.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_directory)


Initial test accuracy on class 0: 83.4%
-------------------------------------------------
Round 1 for class 0
-------------------------------------------------


ValueError: too many values to unpack (expected 2)

### Margin Sampling

A very basic strategy to select unlabeled points is margin sampling, where we select samples about which the model is most uncertain by measuring the difference between the probability of the most confident class and the probability of the second-most confident class. Hence, a valid strategy is to select those points in the unlabeled set with the smallest margin of prediction. Specifically, let $z = f(x)$ be the output from the model with class $i$ having score $z_i$. By applying a softmax, we obtain probabilities that we can use: 

$$\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$ 

Defining

$$\lambda_1 = \text{argmax}_i \sigma(z)_i$$

and

$$\lambda_2 = \text{argmax}_{i \neq \lambda_1} \sigma(z)_i,$$

margin sampling selects a subset based on the following objective:

 $$\text{argmin}_{A \subseteq U, |A| = k} \sum_{x \in A} \left[ \sigma(z)_{\lambda_1} - \sigma(z)_{\lambda_2} \right]$$

Here, we create a instance of `distil.active_learning_strategies.entropy_sampling.MarginSampling` with the same parameters passed to `distil.active_learning_strategies.random_sampling.RandomSampling`.

**Reload Data**

Here we reload, reprocess, and retokenize the data and prepare for a different active learning strategy.

In [ ]:


dataset = load_dataset("fancyzhx/ag_news")

train_dataset = dataset['train']
test_dataset = dataset['test']

# Text preprocessing functions
def to_lower(ex):
    ex['text'] = ex['text'].lower()
    return ex

def remove_punc(ex):
    ex['text'] = ''.join(x for x in ex['text'] if x not in string.punctuation)
    return ex

def to_ascii(ex):
    ex['text'] = ''.join(x for x in unicodedata.normalize('NFD', ex['text']) if unicodedata.category(x) != 'Mn')
    return ex

stopwords = [ "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at", "be", "because", 
             "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do", "does", "doing", "down", "during",
             "each", "few", "for", "from", "further", "had", "has", "have", "having", "he", "he'd", "he'll", "he's", "her", "here", 
             "here's", "hers", "herself", "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into",
             "is", "it", "it's", "its", "itself", "let's", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or",
             "other", "ought", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "she'd", "she'll", "she's", "should", 
             "so", "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves", "then", "there", "there's",
             "these", "they", "they'd", "they'll", "they're", "they've", "this", "those", "through", "to", "too", "under", "until", "up",
             "very", "was", "we", "we'd", "we'll", "we're", "we've", "were", "what", "what's", "when", "when's", "where", "where's",
             "which", "while", "who", "who's", "whom", "why", "why's", "with", "would", "you", "you'd", "you'll", "you're", "you've",
             "your", "yours", "yourself", "yourselves" ]

def remove_stopwords(ex):
    ex['text'] = ' '.join(x for x in ex['text'].split() if x not in stopwords)
    return ex

def process_data(data):
    data1 = data.map(remove_punc)
    data2 = data1.map(to_ascii)
    data3 = data2.map(to_lower)
    data4 = data3.map(remove_stopwords)
    return data4

# Apply processing
training_data = process_data(train_dataset)
testing_data = process_data(test_dataset)


from transformers import AutoTokenizer, get_scheduler, BertConfig, AdamW, BertModel

configuration = BertConfig()
setattr(configuration, 'l1', 512)
setattr(configuration, 'num_classes', 4)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_func(ex):
    return tokenizer(ex['text'], padding='max_length', truncation=True)

# Removes unnecessary columns and renames columns to match the model specification
train1 = training_data.map(tokenize_func, batched=True)
test1 = testing_data.map(tokenize_func, batched=True)
train2 = train1.remove_columns("text")
test2 = test1.remove_columns("text")
training_dataset = train2.rename_column("label", "labels")
testing_dataset = test2.rename_column("label", "labels")

training_dataset.set_format("torch")
testing_dataset.set_format("torch")

# Define the number of active learning rounds to conduct, the budget, and the number of classes in AGN dataset
nclasses = 4
n_rounds = 6
budget = 300

train_size_pc     = 200
test_size_pc      = 1000
unlabeled_size_pc = 5000

# Ensure reproducibility
np.random.seed(42)

# Generate random index lists for subset creation
full_train_subset = []
full_unlabeled_subset = []
full_test_subset = []

for class_num in range(nclasses):
    train_idx_pool = []
    for train_idx in range(len(training_dataset)):
        if training_dataset[train_idx]["labels"].item() == class_num:
            train_idx_pool.append(train_idx)
    test_idx_pool = []
    for test_idx in range(len(testing_dataset)):
        if testing_dataset[test_idx]["labels"].item() == class_num:
            test_idx_pool.append(test_idx)

    # Get a random subset from each pool
    print(f"Class {class_num} - Training pool size: {len(train_idx_pool)}, Testing pool size: {len(test_idx_pool)}")
    train_idx_subset = np.random.choice(train_idx_pool, size=train_size_pc, replace = False).tolist()
    remaining_idx = list(set(train_idx_pool) - set(train_idx_subset))
    unlabeled_idx_subset = np.random.choice(remaining_idx, size=unlabeled_size_pc, replace = False).tolist()
    test_idx_subset = np.random.choice(test_idx_pool, size=test_size_pc, replace = False).tolist()
    full_train_subset.extend(train_idx_subset)
    full_unlabeled_subset.extend(unlabeled_idx_subset)
    full_test_subset.extend(test_idx_subset)


# Create the splits
agn_train     = Subset(training_dataset, full_train_subset)
agn_unlabeled = Subset(training_dataset, full_unlabeled_subset)
testing_dataset = Subset(testing_dataset, full_test_subset)

model = CustomBERTModel(configuration)
state_dict = torch.load(model_directory)
model.load_state_dict(state_dict)

Some weights of the model checkpoint at prajjwal1/bert-small were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


<All keys matched successfully>

In [ ]:

from copy import deepcopy

initial_agn_train = deepcopy(agn_train)
initial_unlabeled = deepcopy(agn_unlabeled)
# Loop over each class
for target_class in range(nclasses):
    print(f"\n========= Class {target_class} =========")

    # Restrict test set to only this class
    class_test_indices = [i for i in range(len(testing_dataset)) if testing_dataset[i]['labels'].item() == target_class]
    class_test_set = Subset(testing_dataset, class_test_indices)

    # Reinitialize and load the pretrained model
   
    model = CustomBERTModel(configuration)
    state_dict = torch.load(model_directory)
    model.load_state_dict(state_dict)

    # Set up training args again
    args = {'lr':1e-4, 'batch_size':5, 'n_epoch': 200, 'optimizer':'adam'}
    agn_train = deepcopy(initial_agn_train)
    dt = data_train(agn_train, model, args)

    # Create AL strategy instance
    strategy_args = {'batch_size' : 5}
    agn_unlabeled = deepcopy(initial_unlabeled)
    strategy = MarginSampling(agn_train, LabeledToUnlabeledDataset(agn_unlabeled), model, nclasses, strategy_args)
    strategy.update_model(model)

    # Track accuracy across rounds
    acc = np.zeros(n_rounds)
    acc[0] = dt.get_acc_on_set(class_test_set)
    print(f"Initial test accuracy on class {target_class}: {round(acc[0]*100, 2)}%")

    # Active Learning Loop
    for rd in range(1, n_rounds):
        print('-------------------------------------------------')
        print(f'Round {rd} for class {target_class}')
        print('-------------------------------------------------')

        # Disable augmentation for selection
        idx = strategy.select(budget)

        # Add selected samples to training set
        selected_subset = Subset(agn_unlabeled, idx)
        agn_train = ConcatDataset([agn_train, selected_subset])

        # Update unlabeled pool
        remaining_indices = list(set(range(len(agn_unlabeled))) - set(idx))
        agn_unlabeled = Subset(agn_unlabeled, remaining_indices)

        print("Number of training samples:", len(agn_train))

        # Update strategy and training data
        strategy.update_data(agn_train, LabeledToUnlabeledDataset(agn_unlabeled))
        dt.update_data(agn_train)

        # Retrain model
        clf = dt.train()
        strategy.update_model(clf)

        # Evaluate on current class only
        acc[rd] = dt.get_acc_on_set(class_test_set)
        print(f"Accuracy on class {target_class}: {round(acc[rd]*100, 2)}%")

    print(f"Training completed for class {target_class}")

    # Save accuracy results
    dir = os.path.join('results', 'AGN', 'baseline')
    os.makedirs(dir, exist_ok=True)
    save_path = os.path.join(dir, f'class_{target_class}_marginSampling.txt')
    with open(save_path, 'w') as f:
        for a in acc:
            f.write(f"{a}\n")
    print(f"Saved class-wise accuracy to: {save_path}")

Initial Testing accuracy: 57.83
-------------------------------------------------
Round 1
-------------------------------------------------
Number of training points - 900
Training..
5 training accuracy: 0.43555555555555553
10 training accuracy: 0.4955555555555556
15 training accuracy: 0.5266666666666666
20 training accuracy: 0.5522222222222222
25 training accuracy: 0.5533333333333333
30 training accuracy: 0.5533333333333333
35 training accuracy: 0.5655555555555556
40 training accuracy: 0.5688888888888889
45 training accuracy: 0.5444444444444444
50 training accuracy: 0.5555555555555556
55 training accuracy: 0.57
60 training accuracy: 0.5855555555555556
65 training accuracy: 0.6
70 training accuracy: 0.58
75 training accuracy: 0.5988888888888889
80 training accuracy: 0.5888888888888889
85 training accuracy: 0.5866666666666667
90 training accuracy: 0.5888888888888889
95 training accuracy: 0.61
100 training accuracy: 0.5933333333333334
105 training accuracy: 0.5933333333333334
110 trainin

## Visualizing the Results

In [ ]:

dir = os.path.join('results', 'AGN', 'baseline')
print(train_size_pc)

# X-axis: number of labeled samples after each round
x_axis = np.array([(train_size_pc*4) + budget * i for i in range(n_rounds)])

# Strategies to compare
strategies = ['random', 'entropy', 'badge', 'leastConfidence', 'marginSampling']
colors = {
    'random': 'r',
    'entropy': 'g',
    'badge': 'c',
    'glister': 'b',
    'leastConfidence': 'm',  # magenta
    'marginSampling': 'y'    # yellow
}
labels = {'random': 'RANDOM', 'entropy': 'UNCERTAINTY', 'badge': 'BADGE', 'leastConfidence': 'LEAST CONFIDENCE', 'marginSampling': 'MARGIN SAMPLING'}

# Plot class-wise curves
for cls in range(nclasses):
    plt.figure(figsize=(8, 5))
    for strat in strategies:
        file_path = os.path.join(dir, f'class_{cls}_{strat}.txt')
        if not os.path.exists(file_path):
            print(f"Missing file: {file_path}")
            continue

        with open(file_path, 'r') as f:
            acc_list = [round(float(line.strip()) * 100, 2) for line in f]
            plt.plot(x_axis, acc_list, label=labels[strat], marker='o', color=colors[strat])

    plt.title(f'Class {cls} Accuracy Over AL Rounds - MNIST')
    plt.xlabel('Number of Labeled Images')
    plt.ylabel('Test Accuracy (%)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


NameError: name 'train_size_pc' is not defined